## Setup and Dependencies

In [1]:
import base64
import json
import os
import time
from pathlib import Path
from typing import Any

import numpy as np
from numpy.linalg import norm
from openai import AzureOpenAI
from dotenv import load_dotenv
from PIL import Image

# Load environment variables
load_dotenv()

# Initialize Azure OpenAI client
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-08-01-preview"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

# Text Embedding Similarity

This section implements a text embedding-based intent classifier using Azure OpenAI embeddings to determine whether input text represents **confirmation** or **denial** intent.

## How It Works

1. **Predefined Phrases**: The function uses curated lists of common confirmation and denial phrases in both English and Tagalog
2. **Text Embeddings**: Uses Azure OpenAI's `text-embedding-3-small` model to convert text into semantic vector representations
3. **Cosine Similarity**: Calculates similarity between the input text embedding and all predefined phrase embeddings
4. **Intent Classification**: Determines the most likely intent based on average similarity scores
5. **Confidence Scoring**: Provides a confidence metric based on the difference between confirmation and denial scores

## Key Features

- **Bilingual Support**: English and Tagalog phrases
- **Semantic Understanding**: Uses embeddings rather than keyword matching for better accuracy
- **Top Matches**: Returns the most similar phrases for interpretability
- **Confidence Scores**: Provides both individual scores and overall confidence

## Use Cases

- Validating user responses in conversational interfaces
- Analyzing extracted text from documents for confirmation/denial patterns
- Intent classification for chatbots and virtual assistants
- Sentiment analysis for yes/no questions

## Save Embeddings to JSON

Generate and save the confirmation and denial phrase embeddings to a local JSON file for reuse.

In [28]:
async def save_embeddings_to_json(
    phrases: list[str],
    output_path: str,
    model: str = "text-embedding-3-small"
) -> str:
    """
    Generate and save embeddings for confirmation and denial phrases to a JSON file.
    
    Args:
        phrases: List of phrases to generate embeddings for
        output_path: Path where the JSON file will be saved
        model: Azure OpenAI embedding model deployment name (default: "text-embedding-3-small")
    
    Returns:
        Path to the saved JSON file
    """
    print(f"Generating embeddings for {len(phrases)} phrases...")
    embeddings = await _get_embedding(phrases, model)
    
    # Create output directory if it doesn't exist
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    
    # Save to JSON file
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(embeddings, f, indent=2, ensure_ascii=False)
    

def load_embeddings_from_json(json_path: str) -> dict[str, Any]:
    """
    Load precomputed embeddings from a JSON file.
    
    Args:
        json_path: Path to the JSON file containing embeddings
    
    Returns:
        Dictionary with confirmation and denial phrases and their embeddings
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    return data


# Define the same phrases used in the main function
CONFIRMATION_PHRASES = [
    # English
    "yes", "yeah", "yup", "correct", "right", "true", "agreed",
    "absolutely", "definitely", "certainly", "sure", "indeed", "confirmed",
    "affirmative", "okay", "ok", "alright", "fine", "good",
    # Tagalog
    "oo", "opo", "tama", "sige", "ayos", "okay lang", "go", "okey",
    "tama yan", "oo nga", "gusto ko"
]

DENIAL_PHRASES = [
    # English
    "no", "nah", "nope", "incorrect", "wrong", "false", "disagreed",
    "absolutely not", "definitely not", "certainly not", "not sure", "not really", "unconfirmed",
    "negative", "not okay", "not ok", "not alright", "not fine", "not good",
    # Tagalog
    "hindi", "hindi po", "mali", "huwag", "hindi ayos", "hindi okay", "wag", "hindi okey",
    "mali yan", "hindi nga", "ayaw ko"
]

# Generate and save embeddings
output_file = await save_embeddings_to_json(
    phrases=DENIAL_PHRASES,
    output_path="./denial_embeddings.json"
)

# print(f"\n✓ Embeddings successfully saved to: {output_file}")

Generating embeddings for 30 phrases...


In [29]:
async def openai_text_embedding_similarity(
    input_text: str,
    model: str = "text-embedding-3-small",
    top_k: int = 3
) -> dict[str, Any]:
    """
    Calculate cosine similarity between input text and predefined confirmation/denial phrases.
    
    Uses Azure OpenAI text embeddings to compute semantic similarity scores for:
    - Confirmation intent (e.g., "yes", "oo", "tama")
    - Denial intent (e.g., "no", "hindi", "mali")
    
    Args:
        input_text: The text to analyze (e.g., user response, extracted text)
        model: Azure OpenAI embedding model deployment name
        top_k: Number of top similar phrases to return per intent
    
    Returns:
        Dictionary containing:
        - confirmation_score: Average cosine similarity to confirmation phrases (0-1)
        - denial_score: Average cosine similarity to denial phrases (0-1)
        - predicted_intent: "confirmation" or "denial" based on higher score
        - confidence: Confidence level (difference between scores)
        - top_confirmation_matches: Top-k most similar confirmation phrases
        - top_denial_matches: Top-k most similar denial phrases
    """
    
    CONFIRMATION_PHRASES = [
        # English
        "yes", "yeah", "yup", "correct", "right", "true", "agreed",
        "absolutely", "definitely", "certainly", "sure", "indeed", "confirmed",
        "affirmative", "okay", "ok", "alright", "fine", "good",
        # Tagalog
        "oo", "opo", "tama", "sige", "ayos", "okay lang", "go", "okey",
        "tama yan", "oo nga", "gusto ko"
    ]

    DENIAL_PHRASES = [
        # English
        "no", "nah", "nope", "incorrect", "wrong", "false", "disagreed",
        "absolutely not", "definitely not", "certainly not", "not sure", "not really", "unconfirmed",
        "negative", "not okay", "not ok", "not alright", "not fine", "not good",
        # Tagalog
        "hindi", "hindi po", "mali", "huwag", "hindi ayos", "hindi okay", "wag", "hindi okey",
        "mali yan", "hindi nga", "ayaw ko"
    ]
    
    # Normalize input text
    input_text = input_text.strip().lower()
    
    # Get embedding for input text (batch call with single item)
    input_embeddings = await _get_embedding([input_text], model)
    input_embedding = input_embeddings[0]
    
    # Get embeddings for all confirmation phrases (batch call)
    confirmation_embeddings = load_embeddings_from_json("./confirmation_embeddings.json")
    
    # Get embeddings for all denial phrases (batch call)
    denial_embeddings = load_embeddings_from_json("./denial_embeddings.json")
    
    # Calculate cosine similarities
    confirmation_similarities = [
        _cosine_similarity(input_embedding, emb) 
        for emb in confirmation_embeddings
    ]
    
    denial_similarities = [
        _cosine_similarity(input_embedding, emb) 
        for emb in denial_embeddings
    ]
    
    # Calculate average scores
    confirmation_score = float(np.mean(confirmation_similarities))
    denial_score = float(np.mean(denial_similarities))
    
    # Get top-k matches for each intent
    confirmation_matches = sorted(
        zip(CONFIRMATION_PHRASES, confirmation_similarities),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]
    
    denial_matches = sorted(
        zip(DENIAL_PHRASES, denial_similarities),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]
    
    # Determine predicted intent
    predicted_intent = "confirmation" if confirmation_score > denial_score else "denial"
    confidence = abs(confirmation_score - denial_score)
    
    return {
        "input_text": input_text,
        "confirmation_score": round(confirmation_score, 4),
        "denial_score": round(denial_score, 4),
        "predicted_intent": predicted_intent,
        "confidence": round(confidence, 4),
        "top_confirmation_matches": [
            {"phrase": phrase, "similarity": round(sim, 4)}
            for phrase, sim in confirmation_matches
        ],
        "top_denial_matches": [
            {"phrase": phrase, "similarity": round(sim, 4)}
            for phrase, sim in denial_matches
        ]
    }


async def _get_embedding(text: list[str], model: str) -> list[list[float]]:
    """
    Get text embeddings from Azure OpenAI using aiohttp.
    
    Args:
        text: List of input texts to embed
        model: Embedding model deployment name
    
    Returns:
        List of embedding vectors (one per input text)
    """
    endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
    api_key = os.getenv("AZURE_OPENAI_API_KEY")
    api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2023-05-15")
    
    url = f"{endpoint}/openai/deployments/{model}/embeddings?api-version={api_version}"
    
    payload = {"input": text}
    headers = {"api-key": api_key, "Content-Type": "application/json"}
    
    import aiohttp
    
    async with aiohttp.ClientSession() as session:
        async with session.post(url, json=payload, headers=headers) as response:
            response.raise_for_status()
            result = await response.json()
            
    # Extract embeddings from response
    return [item["embedding"] for item in result["data"]]


def _cosine_similarity(vec1: list[float], vec2: list[float]) -> float:
    """
    Calculate cosine similarity between two vectors.
    
    Args:
        vec1: First embedding vector
        vec2: Second embedding vector
    
    Returns:
        Cosine similarity score (0-1, where 1 is most similar)
    """
    v1 = np.array(vec1)
    v2 = np.array(vec2)
    return float(np.dot(v1, v2) / (norm(v1) * norm(v2)))

## Test Text Embedding Similarity

Test the function with various confirmation and denial phrases in English and Tagalog.

In [30]:
# Test cases with various inputs
test_inputs = [
    # English confirmations
    "yes",
    "yeah, that's right",
    "definitely correct",
    
    # English denials
    "no",
    "absolutely not",
    "that's wrong",
    
    # Tagalog confirmations
    "oo",
    "tama yan",
    "opo, sang-ayon ako",
    
    # Tagalog denials
    "hindi",
    "mali yan",
    "ayaw ko",
    
    # Ambiguous cases
    "maybe",
    "I'm not sure",
    "okay lang"
]

print("=" * 80)
print("TEXT EMBEDDING SIMILARITY TESTS")
print("=" * 80)

for test_text in test_inputs:
    print(f"\nInput: '{test_text}'")
    print("-" * 80)
    
    result = await openai_text_embedding_similarity(test_text)
    
    print(f"Predicted Intent: {result['predicted_intent'].upper()}")
    print(f"Confidence: {result['confidence']:.4f}")
    print(f"\nScores:")
    print(f"  Confirmation: {result['confirmation_score']:.4f}")
    print(f"  Denial: {result['denial_score']:.4f}")
    
    print(f"\nTop Confirmation Matches:")
    for match in result['top_confirmation_matches']:
        print(f"  - '{match['phrase']}': {match['similarity']:.4f}")
    
    print(f"\nTop Denial Matches:")
    for match in result['top_denial_matches']:
        print(f"  - '{match['phrase']}': {match['similarity']:.4f}")
    
    print()

print("=" * 80)

TEXT EMBEDDING SIMILARITY TESTS

Input: 'yes'
--------------------------------------------------------------------------------
Predicted Intent: CONFIRMATION
Confidence: 0.1028

Scores:
  Confirmation: 0.3701
  Denial: 0.2673

Top Confirmation Matches:
  - 'yes': 1.0000
  - 'yeah': 0.7554
  - 'sure': 0.6713

Top Denial Matches:
  - 'no': 0.6718
  - 'nope': 0.4622
  - 'nah': 0.4093


Input: 'yeah, that's right'
--------------------------------------------------------------------------------
Predicted Intent: CONFIRMATION
Confidence: 0.1093

Scores:
  Confirmation: 0.3553
  Denial: 0.2460

Top Confirmation Matches:
  - 'yeah': 0.5360
  - 'alright': 0.4817
  - 'indeed': 0.4747

Top Denial Matches:
  - 'nope': 0.3559
  - 'wrong': 0.3428
  - 'nah': 0.3269


Input: 'definitely correct'
--------------------------------------------------------------------------------
Predicted Intent: CONFIRMATION
Confidence: 0.0630

Scores:
  Confirmation: 0.3530
  Denial: 0.2900

Top Confirmation Matches:
  